## Bronze Layer to Silver Layer

In [1]:
import polars as pl
from polars import col

import duckdb

### Ingest from Bronze

In [2]:
con = duckdb.connect("../burger.db")

# Control table tracks which bronze records have already been run through this transform
con.execute("""
    CREATE TABLE IF NOT EXISTS bronze_silver_control (
        item_id VARCHAR,
        loaded_at DATE,
        processed_at TIMESTAMP
    )
""")

# Only pull bronze records that haven't been processed yet
df_bronze = con.execute("""
    SELECT b.*
    FROM bronze b
    ANTI JOIN bronze_silver_control c
      ON b.item_id = c.item_id AND b.loaded_at = c.loaded_at
""").pl()

#### Transformations

In [3]:
# Filter out incorrect and incomplete records
df_staging = df_bronze.filter((col('condition') == 'Graded') & 
                              (col('currency') == "USD") &
                              (col('shipping_cost').is_not_null()))

# Ensure Uniqueness
df_staging = df_staging.unique(['item_id', 'loaded_at'])

# Title to uppercase for consistency
df_staging = df_staging.with_columns(
    col('title').str.to_uppercase()
)

# Ensure PSA 10 in title
df_staging = df_staging.filter((col('title').str.contains('PSA')) & (col('title').str.contains('10')))


# Throw out high outliers (skip when there's nothing to compute a median over)
if df_staging.height > 0:
    med = df_staging["total_cost"].median()
    mad = (df_staging["total_cost"] - med).abs().median()

    k = 3
    upper = med + k * mad

    # only an upper bound — nothing on the low side is ever dropped. Throw out nulls in shipping_cost
    df_staging = df_staging.filter(col("total_cost") <= upper)




#### Load to Silver Layer

In [4]:
# Create silver table if it doesn't exist yet (matches df_staging schema)
con.execute("CREATE TABLE IF NOT EXISTS silver AS SELECT * FROM df_staging LIMIT 0")

# Only insert rows whose (item_id, loaded_at) isn't already in silver
con.execute("""
    INSERT INTO silver
    SELECT s.*
    FROM df_staging s
    ANTI JOIN silver t
      ON s.item_id = t.item_id AND s.loaded_at = t.loaded_at
""")

# Mark every bronze record we pulled this run as processed, whether or not it made it into silver
con.execute("""
    INSERT INTO bronze_silver_control
    SELECT item_id, loaded_at, now()
    FROM df_bronze
""")

In [5]:
con.close()